# Stranger Things — Best & Worst Season Keywords (TF-IDF)
Left = Best season (Green) | Right = Worst season (Red)

In [ ]:
# Install: pip install vaderSentiment matplotlib pandas nltk scikit-learn
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import TfidfVectorizer
import nltk
import re

nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

## Step 1: Stopwords

In [ ]:
STOP_WORDS = list(stopwords.words("english"))
CUSTOM_STOP = [
    "show", "season", "one", "like", "just", "get", "got", "even",
    "really", "also", "much", "would", "could", "still", "think",
    "know", "time", "make", "made", "way", "good", "great", "watch",
    "watched", "watching", "series", "episode", "episodes", "things",
    "stranger", "characters", "story", "feel", "feels", "felt",
    "best", "first", "every", "dont", "character", "plot", "seasons",
    "ve", "re", "ll", "said", "say", "says", "going", "come", "came"
]
STOP_WORDS = list(set(STOP_WORDS + CUSTOM_STOP))

## Step 2: Load Data & Run VADER

In [ ]:
df = pd.read_csv("combined_data.csv")
df["text"] = df["text"].fillna("")

analyzer = SentimentIntensityAnalyzer()

def get_compound(text):
    if not str(text).strip():
        return 0.0
    return analyzer.polarity_scores(str(text))["compound"]

print("Running polarity scoring...")
df["compound"] = df["text"].apply(get_compound)

order = ["Season 1", "Season 2", "Season 3", "Season 4", "Season 5"]
avg_polarity = df.groupby("season")["compound"].mean().reindex(order)
best_season  = avg_polarity.idxmax()
worst_season = avg_polarity.idxmin()
print(f"Best season:  {best_season}")
print(f"Worst season: {worst_season}")

## Step 3: TF-IDF Keyword Extraction

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    return text

def get_tfidf_keywords(target_season, polarity_filter, n=15):
    season_docs = {}
    for season in order:
        mask = (df["season"] == season) & (df["compound"].apply(polarity_filter))
        combined = " ".join(df[mask]["text"].apply(clean_text))
        season_docs[season] = combined
    corpus = [season_docs[s] for s in order]
    vectorizer = TfidfVectorizer(
        stop_words=STOP_WORDS, max_features=5000,
        ngram_range=(1, 1), min_df=1,
        token_pattern=r"(?u)\b[a-z]{3,}\b",
    )
    matrix = vectorizer.fit_transform(corpus)
    names  = vectorizer.get_feature_names_out()
    idx     = order.index(target_season)
    scores  = matrix[idx].toarray().flatten()
    top_idx = scores.argsort()[-n:]
    return names[top_idx], scores[top_idx]

best_words,  best_scores  = get_tfidf_keywords(best_season,  lambda x: x > 0)
worst_words, worst_scores = get_tfidf_keywords(worst_season, lambda x: x < 0)
print("Keywords extracted!")

## Step 4: Plot Side by Side

In [ ]:
def plot_keywords(ax, words, scores, cmap_name, title):
    norm   = mcolors.Normalize(vmin=min(scores), vmax=max(scores))
    cmap   = plt.get_cmap(cmap_name)
    colors = [cmap(norm(s)) for s in scores]
    bars = ax.barh(words, scores, color=colors, height=0.6, zorder=2)
    for bar, score in zip(bars, scores):
        ax.text(bar.get_width() + max(scores) * 0.01,
                bar.get_y() + bar.get_height() / 2,
                f"{score:.4f}", va="center", ha="left", fontsize=8, color="white")
    ax.set_xlabel("TF-IDF Score", color="white", fontsize=10)
    ax.tick_params(axis="x", colors="white")
    ax.tick_params(axis="y", colors="white", labelsize=9)
    ax.spines[:].set_visible(False)
    ax.set_xlim(0, max(scores) * 1.2)
    ax.set_facecolor("#1a1a2e")
    ax.set_title(title, color="white", fontsize=11, fontweight="bold", pad=10)

fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(16, 7))
fig.patch.set_facecolor("#1a1a2e")

plot_keywords(ax_left, best_words, best_scores, "Greens",
    f"Top Positive Keywords (TF-IDF)\n{best_season} — Highest Avg Sentiment")
plot_keywords(ax_right, worst_words, worst_scores, "Reds",
    f"Top Negative Keywords (TF-IDF)\n{worst_season} — Lowest Avg Sentiment")

fig.suptitle("Stranger Things — Keyword Analysis by Sentiment Extremes",
             color="white", fontsize=14, fontweight="bold", y=1.05)
plt.tight_layout()
plt.subplots_adjust(top=0.88)
plt.savefig("keywords_best_worst.png", dpi=150, bbox_inches="tight", facecolor="#1a1a2e")
print("\nSaved: keywords_best_worst.png")
plt.show()